# Homework 5a — OpenTelemetry Monitoring

This notebook documents my solution to the **Module 5a homework of [LLM Zoomcamp](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/05-monitoring)** by DataTalksClub.

## What's covered
Manual instrumentation of an existing RAG pipeline using OpenTelemetry — creating traces, capturing token metrics as span attributes, measuring span duration, persisting traces to SQLite, and querying the stored data.

## Stack used

| Component | Tool |
|---|---|
| Tracing | OpenTelemetry SDK |
| Span export | ConsoleSpanExporter, SQLiteSpanExporter |
| RAG pipeline | `RAGBase` (from `rag_helper`) |
| LLM provider | Groq (`openai/gpt-oss-120b`) |
| Index | `minsearch.Index` (pre-built in `starter.py`) |

> API keys are loaded from `.env` via `python-dotenv`.

## Download resoures

In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget -nc {PREFIX}/rag_helper.py
!wget -nc {PREFIX}/starter.py

File 'rag_helper.py' already there; not retrieving.

File 'starter.py' already there; not retrieving.



## Setup — OpenAI-compatible Client

The RAG pipeline uses Groq via the OpenAI-compatible API.

In [4]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from pathlib import Path

path = Path.cwd().parent/".env"
print(load_dotenv(path))

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
    )

True


## Test the RAG Pipeline

A quick smoke test to confirm the RAG pipeline is working before we add tracing.

Modified `client=OpenAI()` -> 

```python
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
    )
```
 and add `model="openai/gpt-oss-120b` in `RAGBase()`

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop is just a `while‑True` that keeps sending the current conversation back to the model, runs any tool calls that the model asked for, and then checks whether the model asked for another tool.  

**What it does step‑by‑step**

1. **Start a loop**  
   ```python
   it = 1
   while True:
   ```
   An iteration counter (`it`) is optional – it just lets you see how many round‑trips have happened.

2. **Call the model**  
   ```python
   response = openai_client.responses.create(
       model=model,
       input=messages,
       tools=[search_tool],
   )
   ```
   The current `messages` list (developer prompt, user question, all prior model replies and any tool‑output) is sent as the input.

3. **Append the raw response to the history**  
   ```python
   messages.extend(response.output)
   ```

4. **Process each item in the response**  
   * If the item is a `function_call`, the code:
   * parses the JSON arguments, runs the appropriate Python function (`make_call`),  
   * cr

# OpenTelemetry Setup

Before instrumenting the RAG pipeline, two minimal examples confirm the tracer is working.

### Example 1 — Basic span with a single attribute

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

with tracer.start_as_current_span("my_operation") as span:
    # your code here
    span.set_attribute("my_key", "my_value")

{
    "name": "my_operation",
    "context": {
        "trace_id": "0xe0168656393f7d18e3636783de7c4678",
        "span_id": "0xe7f6829a524a6994",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T17:03:11.826652Z",
    "end_time": "2026-07-20T17:03:11.826799Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "92d85137-7a4f-4e52-9895-31421dcfe032",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


### Example 2 — Span with multiple attributes and console export

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# 1. Initialize the OpenTelemetry infrastructure
# Create the central configuration provider
provider = TracerProvider()

# Wire up a simple span processor that prints every finished span to the console
processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(processor)

# # Register the provider globally so other modules can access it
trace.set_tracer_provider(provider)

# 2. Get a tracer instance for your app
tracer = trace.get_tracer("my-traced-application")

# 3. Execute the traced code block
print("--- Starting operation ---")
with tracer.start_as_current_span("my_operation") as span:
    # Your business logic lives inside this block
    print("Performing some work inside the span...")
    
    # Attach custom metrics or metadata to this specific span
    span.set_attribute("my_key", "my_value")
    span.set_attribute("user_id", 42)

print("--- Operation finished! Check console output below ---")

--- Starting operation ---
Performing some work inside the span...
{
    "name": "my_operation",
    "context": {
        "trace_id": "0xaf6d3b8597a3fb1ac1eac9f5ea14a37b",
        "span_id": "0x2414dbdaa58cd336",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T17:11:13.734448Z",
    "end_time": "2026-07-20T17:11:13.734505Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value",
        "user_id": 42
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "f5a1cd76-861c-49f6-b89a-f7625f71fb09",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
--- Operation finished! Check console output below ---


## Q1 — First trace

Instrument the RAG pipeline by subclassing `RAGBase` and wrapping each method in a named span: `rag` (parent), `search` (child), and `llm` (child).

**How many spans does a single RAG run produce?**

In [3]:
import time

from tqdm.auto import tqdm
from rag_helper import RAGBase

In [4]:
class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
             
            return response

In [7]:
# Q1) How many spans does the trace produce?

from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x1279f294aa0473ac7cfbd7ef680689cd",
        "span_id": "0x07a64a48bf87e827",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x741428f511f71623",
    "start_time": "2026-07-20T17:12:54.996671Z",
    "end_time": "2026-07-20T17:12:55.004379Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "f5a1cd76-861c-49f6-b89a-f7625f71fb09",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x1279f294aa0473ac7cfbd7ef680689cd",
        "span_id": "0x95a08418ec817df5",
        "trace_state": "[]"
    },
    "kind": "SpanKind

'The loop is just a simple\u202f`while True`\u202fthat repeats the request‑response cycle and stops only when the model no longer asks for a tool.\n\n```python\nit = 1\nwhile True:\n    print(f"iteration #{it}…")\n    has_function_calls = False          # reset flag each turn\n\n    # 1️⃣ Call the model with the current message history\n    response = openai_client.responses.create(\n        model="gpt‑5.4‑mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    # 2️⃣ Add everything the model returned to the history\n    messages.extend(response.output)\n\n    # 3️⃣ Walk through the new items\n    for item in response.output:\n        if item.type == "function_call":\n            # The model wants a tool → run it\n            call_output = make_call(item)          # run the search, serialize result\n            messages.append(call_output)           # give the result back to the model\n            has_function_calls = True              # we’ll need another turn\n    

Q1) How many spans does the trace produce?

- First Span: "name": `"search"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

- Second Span: "name": `"llm"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

- Third Span: "name": `"rag"` has trace_id: "0x899c38e5194973aecf24ee06bcaf7df3"

<font color='green'>Total spans by the trace: 3</font>

## Q2 — Capturing metrics as span attributes

Add token usage counters (`input_tokens`, `output_tokens`) as attributes on the `llm` span by reading `response.usage`.

In [8]:
class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            # Q2. Capture token metrics from the response
            if hasattr(response, 'usage') and response.usage:
                span.set_attribute("input_tokens", response.usage.input_tokens)
                span.set_attribute("output_tokens", response.usage.output_tokens)
             
            return response

In [9]:
# Q2) How many input tokens do we see?

from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x75f0db967995cd6e7255b3b4824388f1",
        "span_id": "0x4450ea455caf85f2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x200c64e19b67a7cd",
    "start_time": "2026-07-20T17:14:45.210621Z",
    "end_time": "2026-07-20T17:14:45.218814Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "f5a1cd76-861c-49f6-b89a-f7625f71fb09",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x75f0db967995cd6e7255b3b4824388f1",
        "span_id": "0xfd943304f6377182",
        "trace_state": "[]"
    },
    "kind": "SpanKind

'The agentic loop is just a\u202f`while‑True`\u202fcycle that repeatedly sends the current conversation (the **messages** list) to the LLM, looks at the response, and decides whether another round is needed.\n\n1. **Send the request** –\u202f`openai_client.responses.create` is called with the full `messages` history and the available tools.  \n2. **Add the response to the history** –\u202f`messages.extend(response.output)` so the model will see its own output on the next turn.  \n3. **Inspect the output** –\u202fthe loop iterates over `response.output`.  \n   * If an item’s `type` is `"function_call"` the code  \n        * prints the call,  \n        * runs the corresponding Python function (`make_call`),  \n        * appends the function‑call result back into `messages`, and  \n        * sets `has_function_calls = True`.  \n   * If the item is a normal `"message"` it just prints the assistant’s text.  \n4. **Check the exit flag** – after processing all items the loop does  \n\n   ```p

 Q2. How many input tokens do we see?

```bash
 "attributes": {
        "input_tokens": 7174,
        "output_tokens": 624
    }
```
<font color='green'>Answer: ~7000 </font>
>

## Q3 — Span timing

Using the timestamps from the Q1 trace output, calculate the duration of the `search` and `llm` spans.

From Q1.

```bash
"name": "llm",
    "context": {
        "trace_id": "0x9762848f288bb536e0b36c5af5db392b",
        "span_id": "0x21bf5dbd57efd9ea",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2bbfdf59ad5b06e8",
    "start_time": "2026-07-20T05:29:54.550763Z",
    "end_time": "2026-07-20T05:29:56.696679Z",
```

In [10]:
from datetime import datetime

# Timestamps from the Q1 trace output
search_start = datetime.strptime("2026-07-20T05:29:54.542526Z", "%Y-%m-%dT%H:%M:%S.%fZ")
search_end = datetime.strptime("2026-07-20T05:29:54.548211Z", "%Y-%m-%dT%H:%M:%S.%fZ")

llm_start = datetime.strptime("2026-07-20T05:29:54.550763Z", "%Y-%m-%dT%H:%M:%S.%fZ")
llm_end = datetime.strptime("2026-07-20T05:29:56.696679Z", "%Y-%m-%dT%H:%M:%S.%fZ")

# Calculation
search_ms = (search_end - search_start).total_seconds() * 1000
llm_ms = (llm_end - llm_start).total_seconds() * 1000

print(f"Search span duration: {search_ms:.2f} ms")
print(f"LLM span duration: {llm_ms:.2f} ms")

Search span duration: 5.68 ms
LLM span duration: 2145.92 ms


Q3. For a typical query, roughly how long does the LLM call take?

<font color=green>Answer: > 2000ms </font>

## Q4 — Saving traces to SQLite

Instead of printing spans to the console, we write them to a local SQLite database with a custom `SQLiteSpanExporter`. The schema stores name, timing, token counts, and cost.

Need to restart the notebook to resolve -> `Overriding of current TracerProvider is not allowed`

### Custom SQLite exporter

Setup `traces.db`

In [1]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

traces.db store span:
- name
- start_time
- end_time
- input_tokens
- output_tokens
- cost

### Set up tracer with SQLite export

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
processor = SimpleSpanProcessor(SQLiteSpanExporter("traces.db")) # Replace ConsoleSpanExporter
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

### Run the instrumented RAG (with cost tracking)

In [3]:
from rag_helper import RAGBase 

class RAGTraced(RAGBase):

    def rag(self, query):
        # Create the top-level "parent" span for the entire RAG pipeline
        with tracer.start_as_current_span("rag") as span:
            # Call the original rag method from the parent class
            return super().rag(query)

    def search(self, query, num_results=5):
        # Create a "child" span specifically for the index search step
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        # Create a "child" span specifically for the LLM API call step
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            # Q2. Capture token metrics from the response
            if hasattr(response, 'usage') and response.usage:
                span.set_attribute("input_tokens", response.usage.input_tokens)
                span.set_attribute("output_tokens", response.usage.output_tokens)

                # calculate cost
                input_price_per_million = 0.15 # groq opeanai/gpt-oss-120b # 0.75 
                output_price_per_million = 0.6 # groq opeanai/gpt-oss-120b # 4.50

                input_cost = (response.usage.input_tokens / 1_000_000) * input_price_per_million
                output_cost = (response.usage.output_tokens / 1_000_000) * output_price_per_million
                total_cost = input_cost + output_cost
                span.set_attribute("cost",total_cost)

            return response


# ---------------------- #
from starter import index
from starter import client as llm_client

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a\u202f`while True`\u202fthat repeats the same request‑response cycle over and over, stopping only when the model no longer asks for a tool.\n\nHere’s the essential flow (from **14‑agentic‑loop.md**):\n\n1. **Start the loop** – `while True:` (with an iteration counter `it` just for logging).  \n2. **Call the model** – send the current `messages` list (the whole conversation history) to `openai_client.responses.create(...)`.  \n3. **Process the response** –  \n   * Append every item the model returned (`response.output`) to `messages`.  \n   * If an item is a `function_call`, execute the corresponding tool (`make_call(item)`), append the tool’s output back to `messages`, and set `has_function_calls = True`.  \n   * If an item is a normal `message`, just display it.  \n4. **Check the exit flag** – after iterating over all items, the loop does:\n\n```python\nif has_function_calls == False:\n    break\n```\n\n   *If the model produced **no** function calls in that

### Inspect the stored spans

In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784568343812611000,1784568343823579000,NaN,NaN,NaN
1,llm,1784568343828660000,1784568343832263000,NaN,NaN,NaN
2,rag,1784568343812350000,1784568343833461000,NaN,NaN,NaN
3,search,1784568449691262000,1784568449700930000,NaN,NaN,NaN
4,llm,1784568449702269000,1784568452121940000,7174.0,575.0,0.001421
5,rag,1784568449691016000,1784568452122945000,NaN,NaN,NaN
6,search,1784568582241308000,1784568582249213000,NaN,NaN,NaN
7,llm,1784568582251753000,1784568585182733000,7174.0,674.0,0.001481
8,rag,1784568582241247000,1784568585187547000,NaN,NaN,NaN
9,search,1784568886565225000,1784568886566150000,NaN,NaN,NaN


Q4. Which span names appear in the spans table?

<font color="green">Answer: search, llm, rag </font>

## Q5 — Querying trace data

Compute the total duration grouped by span type (excluding the root `rag` span) to identify which operation consumes the most time.

In [5]:
# ---------------------- #
from starter import index , client as llm_client

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a **`while‑True` loop that repeats the model call until the model no longer asks for a tool**.\n\n1. **Start a turn** –\u202fsend the current `messages` (instructions, user query, any previous tool results) to the model with `openai_client.responses.create(...)`.\n\n2. **Inspect the response** –\u202fthe response can contain:\n   * `function_call` items (the model wants to run a tool)  \n   * `message` items (the model is giving an answer).\n\n3. **If there are function calls**:  \n   * Run each call with `make_call(item)` (which executes the tool and returns a `function_call_output`).  \n   * Append the tool‑output objects to `messages`.  \n   * Set a flag `has_function_calls = True`.\n\n4. **If there are only messages**: the flag stays `False`.\n\n5. **Loop control** –\u202fafter processing all items, the code does:\n\n```python\nif has_function_calls == False:\n    break      # no more tool calls → we’re done\n```\n\nBecause the flag is reset to `False` at 

In [11]:
df = pd.read_sql_query("SELECT * FROM spans", sqlite3.connect("traces.db"))
df['duration_ms'] = (df['end_time'] - df['start_time']) / 1_000_000
summary = df[df['name'] != 'rag'].groupby('name')['duration_ms'].mean().reset_index()
summary

,name,duration_ms
0,llm,9668.7836
1,search,8.2108


Q5) Which span type takes the most total time?

<font color="green">Answer: llm </font>

## Q6 — Token stability across runs

Run the traced RAG multiple times and check whether the input token count stays the same across runs.

In [12]:
# ---------------------- #
from starter import index

query = 'How does the agentic loop keep calling the model until it stops?'
assistant = RAGTraced(
    index=index,
    llm_client=llm_client,
    model="openai/gpt-oss-120b",
)
assistant.rag(query)

'The agentic loop is just a\u202f`while True`\u202fthat repeatedly sends the current message history to the model, runs any tools the model asks for, and then checks whether the model asked for another tool call.\n\n**What the loop does**\n\n1. **Start an iteration** – print the iteration number and reset a flag (`has_function_calls = False`).  \n2. **Call the model** – `openai_client.responses.create(model, input=messages, tools=[search_tool])`.  \n3. **Append the model’s output** to `messages` so the next turn sees everything that has happened.  \n4. **Inspect each output item**:  \n   * If `item.type == "function_call"` →  \n     * Print the call, execute it with `make_call(item)`, append the function‑call‑output to `messages`, and set `has_function_calls = True`.  \n   * If `item.type == "message"` → print the assistant’s text.  \n5. **Increment the iteration counter.**  \n6. **Exit condition** – after processing the turn, `if has_function_calls == False: break`.  \n   *No function

In [13]:
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()

df = df[df['name']=='llm']
df.drop(columns=['start_time','end_time'],axis=0)

,name,input_tokens,output_tokens,cost
1,llm,NaN,NaN,NaN
4,llm,7174.0,575.0,0.001421
7,llm,7174.0,674.0,0.001481
10,llm,7174.0,516.0,0.001386
13,llm,7174.0,458.0,0.001351
16,llm,7174.0,459.0,0.001351


Q6) How much do the input tokens vary across these 4 runs?

<font color="green">Answer: They're identical </font>